# Clip Extractor — Kaggle

Chuyển export `shot` thành logical `clipwindow`, tùy chọn cắt MP4 local và sinh SQL để import sau. Notebook không kết nối database, R2, caption, ASR hoặc embedding. Mặc định metadata-only nên không dùng GPU là đúng thiết kế.


In [ ]:
# Configuration
from __future__ import annotations
import os
from pathlib import Path
INPUT_DIR=Path(os.environ.get('CLIP_INPUT_DIR','/kaggle/input/btc-shots'))
OUTPUT_DIR=Path(os.environ.get('CLIP_OUTPUT_DIR','/kaggle/working/clip_extractor_output'))
SHOT_FILE=Path(os.environ['SHOT_FILE']) if os.environ.get('SHOT_FILE') else next((path for path in (INPUT_DIR/'shot.csv',INPUT_DIR/'shots.csv') if path.is_file()),INPUT_DIR/'shot.csv')
VIDEOS_FILE=os.environ.get('VIDEOS_FILE','') # optional videos.csv/video.txt
SPLIT_THRESHOLD_MS=10_000; MAX_CLIP_DURATION_MS=10_000; STRIDE_MS=8_000; MIN_NEW_WINDOW_GAP_MS=2_000
SAMPLING_FPS=None; MATERIALIZE_FILES=False; VIDEO_CODEC='libx264'; AUDIO_CODEC='aac'; PRESET='veryfast'; CRF=23
FFMPEG_THREADS=2; DRY_RUN=False; OVERWRITE_FILES=False; CLIP_PATH_PREFIX='data/clips'
VIDEO_START=0; VIDEO_END=None; VIDEO_IDS=set(); ALLOW_CHECKPOINT_OVERRIDE=False
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
assert all(x>0 for x in (SPLIT_THRESHOLD_MS,MAX_CLIP_DURATION_MS,STRIDE_MS,MIN_NEW_WINDOW_GAP_MS))
assert SPLIT_THRESHOLD_MS<=MAX_CLIP_DURATION_MS and 0<STRIDE_MS<=MAX_CLIP_DURATION_MS and MIN_NEW_WINDOW_GAP_MS<=STRIDE_MS and 0<=CRF<=51
assert SAMPLING_FPS is None or SAMPLING_FPS>0


In [ ]:
# Standard-library implementation; Pandas/PyArrow are only necessary if SHOT_FILE is Parquet.
from collections import Counter
import csv, hashlib, json, math, re, shutil, subprocess, time, traceback, urllib.parse, urllib.request
from dataclasses import dataclass, asdict
from typing import Any

ERRORS=OUTPUT_DIR/'errors.jsonl'; MANIFEST=OUTPUT_DIR/'clip_manifest.jsonl'; CHECKPOINT=OUTPUT_DIR/'checkpoint.json'
COLUMNS=['clip_id','shot_id','start_ms','end_ms','start_frame_idx','end_frame_idx','sampling_fps','clip_path']
def write_jsonl(path:Path,row:dict[str,Any])->None:
    with path.open('a',encoding='utf-8') as f:f.write(json.dumps(row,ensure_ascii=False,default=str)+'\n')
def fail(stage:str,message:str,**kw:Any)->None: write_jsonl(ERRORS,{'stage':stage,'message':message,**kw})
def command(args:list[str])->subprocess.CompletedProcess[str]: return subprocess.run(args,check=True,text=True,capture_output=True)
def sql(v:Any)->str:
    if v is None:return 'NULL'
    if isinstance(v,float):
        if not math.isfinite(v):raise ValueError('non-finite numeric value')
        return repr(v)
    if isinstance(v,int):return str(v)
    return "'"+str(v).replace("'","''")+"'"
def safe(value:str)->str:
    result=re.sub(r'[^A-Za-z0-9_.-]','_',value)
    if not result:raise ValueError('empty safe path component')
    return result
def input_hash()->str:
    h=hashlib.sha256(json.dumps({k:v for k,v in globals().items() if k in {'SPLIT_THRESHOLD_MS','MAX_CLIP_DURATION_MS','STRIDE_MS','MIN_NEW_WINDOW_GAP_MS','SAMPLING_FPS','MATERIALIZE_FILES','CLIP_PATH_PREFIX'}},sort_keys=True,default=str).encode())
    for p in [SHOT_FILE]+([Path(VIDEOS_FILE)] if VIDEOS_FILE else []):h.update(p.name.encode());h.update(p.read_bytes())
    return h.hexdigest()


In [ ]:
# Input validation. Invalid rows are retained only in errors.jsonl.
@dataclass(frozen=True)
class Shot:
    shot_id:str; video_id:str; shot_index:int; start_ms:int; end_ms:int; start_frame_idx:int; end_frame_idx:int; video_path:str|None=None

def read_table(path:Path)->list[dict[str,Any]]:
    if path.suffix.lower()=='.parquet':
        try:
            import pandas as pd
        except ImportError as exc: raise RuntimeError('Parquet needs pandas/pyarrow; install only if using Parquet') from exc
        return pd.read_parquet(path).to_dict('records')
    with path.open(encoding='utf-8-sig',newline='') as f:return list(csv.DictReader(f))
def load_shots()->list[Shot]:
    required={'shot_id','video_id','shot_index','start_ms','end_ms','start_frame_idx','end_frame_idx'}; rows=read_table(SHOT_FILE)
    if not rows or required-set(rows[0]):raise ValueError(f'shot.csv missing columns: {sorted(required-set(rows[0]) if rows else required)}')
    output=[]; seen=set()
    for number,row in enumerate(rows,2):
        try:
            shot=Shot(str(row['shot_id']).strip(),str(row['video_id']).strip(),int(row['shot_index']),int(row['start_ms']),int(row['end_ms']),int(row['start_frame_idx']),int(row['end_frame_idx']),str(row.get('video_path','')).strip() or None)
            if not shot.shot_id or not shot.video_id or len(shot.shot_id)>15 or len(shot.video_id)>15:raise ValueError('IDs must be nonempty and <=15 chars')
            if (shot.video_id,shot.shot_index) in seen:raise ValueError('duplicate shot_index within video')
            if shot.start_ms<0 or shot.end_ms<=shot.start_ms or shot.start_frame_idx<0 or shot.end_frame_idx<=shot.start_frame_idx:raise ValueError('invalid half-open boundaries')
            seen.add((shot.video_id,shot.shot_index));output.append(shot)
        except Exception as exc:fail('validate_shot',str(exc),row_number=number,row=row)
    output.sort(key=lambda x:(x.video_id,x.shot_index,x.start_ms,x.shot_id))
    for a,b in zip(output,output[1:]):
        if a.video_id==b.video_id and a.end_ms!=b.start_ms:write_jsonl(MANIFEST,{'status':'warning','video_id':a.video_id,'previous_shot_id':a.shot_id,'shot_id':b.shot_id,'warning':'shot_gap' if a.end_ms<b.start_ms else 'shot_overlap'})
    return output


In [ ]:
# Exact backend-compatible window planning and IDs.
CANONICAL=re.compile(r'^(L\d+)_V(\d+)_S(\d+)$',re.I)
def clip_id(shot:Shot,n:int)->str:
    match=CANONICAL.fullmatch(shot.shot_id)
    if match and n<=99:
        candidate=f'{match.group(1).upper()}V{match.group(2)}S{match.group(3)}C{n:02d}'
        if len(candidate)<=15:return candidate
    return 'C'+hashlib.sha1(f'{shot.shot_id}:{n}'.encode()).hexdigest()[:14].upper()
def windows(shot:Shot)->list[tuple[int,int]]:
    if shot.end_ms-shot.start_ms<=SPLIT_THRESHOLD_MS:return [(shot.start_ms,shot.end_ms)]
    values=[];start=shot.start_ms
    while start+MAX_CLIP_DURATION_MS<=shot.end_ms:
        values.append((start,start+MAX_CLIP_DURATION_MS));start+=STRIDE_MS
    tail=(shot.end_ms-MAX_CLIP_DURATION_MS,shot.end_ms)
    if not values or values[-1][1]!=shot.end_ms:
        if len(values)>1 and tail[0]-values[-1][0]<MIN_NEW_WINDOW_GAP_MS:values[-1]=tail
        else:values.append(tail)
    assert values[0][0]==shot.start_ms and values[-1][1]==shot.end_ms
    assert all(a>=shot.start_ms and b<=shot.end_ms and 0<b-a<=MAX_CLIP_DURATION_MS for a,b in values)
    assert all(b[0]<=a[1] for a,b in zip(values,values[1:]))
    return values
def frame_boundary(shot:Shot,boundary:int)->int:
    return shot.start_frame_idx+round((shot.end_frame_idx-shot.start_frame_idx)*(boundary-shot.start_ms)/(shot.end_ms-shot.start_ms))
def plan(shot:Shot)->list[dict[str,Any]]:
    fps=SAMPLING_FPS if SAMPLING_FPS is not None else (shot.end_frame_idx-shot.start_frame_idx)*1000.0/(shot.end_ms-shot.start_ms)
    result=[]
    for index,(start,end) in enumerate(windows(shot),1):
        cid=clip_id(shot,index);logical=f'{CLIP_PATH_PREFIX.strip("/")}/{safe(shot.video_id)}/{safe(cid)}.mp4' if MATERIALIZE_FILES else None
        if logical and len(logical)>200:raise ValueError('clip_path exceeds varchar(200)')
        result.append({'clip_id':cid,'shot_id':shot.shot_id,'video_id':shot.video_id,'shot_index':shot.shot_index,'start_ms':start,'end_ms':end,'start_frame_idx':frame_boundary(shot,start),'end_frame_idx':frame_boundary(shot,end),'sampling_fps':fps,'clip_path':logical,'clip_index':index,'input_start_ms':shot.start_ms,'input_end_ms':shot.end_ms})
    if len({x['clip_id'] for x in result})!=len(result):raise ValueError('internal clip ID conflict')
    return result


In [ ]:
# Optional local video resolution and accurate FFmpeg materialization (CPU by default).
def manifests()->dict[str,str]:
    if not VIDEOS_FILE:return {}
    p=Path(VIDEOS_FILE); rows=[]
    if p.suffix=='.txt':
        for line in p.read_text().splitlines():
            if line.strip():
                a,b=line.split(',',1);rows.append({'video_id':a.strip(),'video_path':b.strip()})
    else:rows=read_table(p)
    return {str(r['video_id']):str(r.get('video_path') or r.get('video_url') or '') for r in rows}
def resolve(source:str,video_id:str)->Path:
    if not source:raise RuntimeError('missing video_path/video manifest')
    parsed=urllib.parse.urlparse(source)
    if parsed.scheme in ('http','https'):
        dest=OUTPUT_DIR/'cache'/f'{safe(video_id)}{Path(parsed.path).suffix or ".mp4"}';dest.parent.mkdir(exist_ok=True)
        if not dest.exists() or not dest.stat().st_size:
            for attempt in range(3):
                try:
                    with urllib.request.urlopen(source,timeout=120) as r,dest.open('wb') as o:shutil.copyfileobj(r,o)
                    if dest.stat().st_size:break
                except Exception:
                    if attempt==2:raise
                    time.sleep(2**attempt)
        return dest
    return Path(source)
def duration_ms(path:Path)->int:
    value=float(command(['ffprobe','-v','error','-show_entries','format=duration','-of','default=nw=1:nk=1',str(path)]).stdout.strip());return round(value*1000)
def materialize(row:dict[str,Any],video:Path)->Path:
    target=OUTPUT_DIR/'clips'/safe(row['video_id'])/f'{safe(row["clip_id"])}.mp4';target.parent.mkdir(parents=True,exist_ok=True)
    if target.exists() and not OVERWRITE_FILES and target.stat().st_size and abs(duration_ms(target)-(row['end_ms']-row['start_ms']))<=150:return target
    target.unlink(missing_ok=True);args=['ffmpeg','-hide_banner','-loglevel','error','-y','-i',str(video),'-ss',f'{row["start_ms"]/1000:.3f}','-t',f'{(row["end_ms"]-row["start_ms"])/1000:.3f}','-map','0:v:0','-map','0:a?']
    if SAMPLING_FPS is not None:args+=['-vf',f'fps={SAMPLING_FPS}']
    args+=['-c:v',VIDEO_CODEC,'-preset',PRESET,'-crf',str(CRF),'-c:a',AUDIO_CODEC,'-movflags','+faststart','-avoid_negative_ts','make_zero','-threads',str(FFMPEG_THREADS),str(target)]
    command(args)
    if not target.exists() or not target.stat().st_size or abs(duration_ms(target)-(row['end_ms']-row['start_ms']))>150:target.unlink(missing_ok=True);raise RuntimeError('invalid materialized MP4')
    return target


In [ ]:
# Pipeline, deterministic artifacts and atomic checkpoint.
def run_pipeline()->dict[str,Any]:
    for p in (ERRORS,MANIFEST):p.unlink(missing_ok=True)
    digest=input_hash(); state={'hash':digest,'completed':[],'rows':[],'mode':MATERIALIZE_FILES}
    if CHECKPOINT.exists():
        state=json.loads(CHECKPOINT.read_text())
        if (state['hash']!=digest or state['mode']!=MATERIALIZE_FILES) and not ALLOW_CHECKPOINT_OVERRIDE:raise RuntimeError('checkpoint hash/mode mismatch')
    all_shots=load_shots(); videos=sorted({x.video_id for x in all_shots});selected=[v for v in videos[VIDEO_START:VIDEO_END] if not VIDEO_IDS or v in VIDEO_IDS]; mapping=manifests(); begin=time.perf_counter();failed=[]
    for video_id in selected:
        if video_id in state['completed']:continue
        group=[x for x in all_shots if x.video_id==video_id]; rows=[]
        try:
            source=None
            if MATERIALIZE_FILES and not DRY_RUN:
                source=resolve(next((x.video_path for x in group if x.video_path),mapping.get(video_id,'')),video_id)
                if not source.exists() or not source.stat().st_size:raise RuntimeError('video missing/empty')
                limit=duration_ms(source)
                if any(x.end_ms>limit+100 for x in group):raise RuntimeError(f'shot exceeds video duration {limit}ms')
            for shot in group:
                for row in plan(shot):
                    tick=time.perf_counter();physical=None
                    if MATERIALIZE_FILES and not DRY_RUN:physical=str(materialize(row,source))
                    row['physical_path']=physical;row['status']='planned' if not MATERIALIZE_FILES or DRY_RUN else 'materialized';row['elapsed_s']=time.perf_counter()-tick;rows.append(row);write_jsonl(MANIFEST,row)
            state['rows']+=rows;state['completed'].append(video_id);tmp=CHECKPOINT.with_suffix('.tmp');tmp.write_text(json.dumps(state,sort_keys=True),encoding='utf-8');tmp.replace(CHECKPOINT)
        except Exception as exc:failed.append(video_id);fail('video',str(exc),video_id=video_id,traceback=traceback.format_exc())
    rows=sorted(state['rows'],key=lambda x:(x['video_id'],x['shot_index'],x['start_ms'],x['clip_id']))
    keys=[(x['clip_id'],x['shot_id'],x['start_ms'],x['end_ms']) for x in rows];key_counts=Counter(keys);id_counts=Counter(x['clip_id'] for x in rows);conflicts=[x for x,key in zip(rows,keys) if key_counts[key]>1 or id_counts[x['clip_id']]>1]
    with (OUTPUT_DIR/'preflight_conflicts.csv').open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=COLUMNS);w.writeheader();w.writerows([{k:x.get(k) for k in COLUMNS} for x in conflicts])
    if conflicts:raise RuntimeError('internal ID/range conflict: SQL not written')
    with (OUTPUT_DIR/'clipwindow.csv').open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=COLUMNS);w.writeheader();w.writerows([{k:('' if x.get(k) is None else x[k]) for k in COLUMNS} for x in rows])
    header='-- Import table shot before this file. Generated by Clip Extractor Kaggle.\n'
    statements=[]
    for x in rows:statements.append('INSERT INTO clipwindow\n  ('+', '.join(COLUMNS)+')\nVALUES ('+', '.join(sql(x.get(k)) for k in COLUMNS)+')\nON CONFLICT DO NOTHING;')
    (OUTPUT_DIR/'insert_clipwindows.sql').write_text(header+'\n'.join(statements)+'\n',encoding='utf-8')
    summary={'videos':len(selected),'shots':len([x for x in all_shots if x.video_id in selected]),'clips':len(rows),'materialize_files':MATERIALIZE_FILES,'failed_videos':failed,'seconds':time.perf_counter()-begin,'ffmpeg_threads':FFMPEG_THREADS,'gpu_used':False,'output_dir':str(OUTPUT_DIR)}
    (OUTPUT_DIR/'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8');shutil.rmtree(OUTPUT_DIR/'cache',ignore_errors=True);shutil.rmtree(OUTPUT_DIR/'clips',ignore_errors=True);return summary


In [ ]:
# Real Kaggle run: attach shot.csv (and video files/manifest only when materializing), set config above, then uncomment.
# print(json.dumps(run_pipeline(),indent=2))


In [ ]:
# Mandatory mock test (metadata twice, negative validation, then optional FFmpeg materialization).
def mock_test()->None:
    global INPUT_DIR,OUTPUT_DIR,SHOT_FILE,MATERIALIZE_FILES,VIDEOS_FILE,CHECKPOINT,ERRORS,MANIFEST
    names=('INPUT_DIR','OUTPUT_DIR','SHOT_FILE','MATERIALIZE_FILES','VIDEOS_FILE','CHECKPOINT','ERRORS','MANIFEST')
    original={name:globals()[name] for name in names}
    try:
        root=Path('/tmp/clip_mock');shutil.rmtree(root,ignore_errors=True);root.mkdir();INPUT_DIR=root;OUTPUT_DIR=root/'out';OUTPUT_DIR.mkdir();SHOT_FILE=root/'shot.csv';CHECKPOINT=OUTPUT_DIR/'checkpoint.json';ERRORS=OUTPUT_DIR/'errors.jsonl';MANIFEST=OUTPUT_DIR/'clip_manifest.jsonl';VIDEOS_FILE=''
        SHOT_FILE.write_text('shot_id,video_id,shot_index,start_ms,end_ms,start_frame_idx,end_frame_idx\nS_BAD,v1,0,0,0,0,1\nL21_V001_S0003,v1,1,80000,105000,800,1050\nweird id,v2,0,0,10000,0,100\n',encoding='utf-8')
        MATERIALIZE_FILES=False;first=run_pipeline();second=run_pipeline();rows=list(csv.DictReader((OUTPUT_DIR/'clipwindow.csv').open()))
        planned=[r for r in rows if r['shot_id']=='L21_V001_S0003'];assert [(int(x['start_ms']),int(x['end_ms'])) for x in planned]==[(80000,90000),(88000,98000),(95000,105000)]
        assert planned[0]['clip_id']=='L21V001S0003C01' and len(planned[0]['clip_id'])<=15 and rows[-1]['clip_id'].startswith('C')
        assert list(rows[0])==COLUMNS and 'ON CONFLICT DO NOTHING' in (OUTPUT_DIR/'insert_clipwindows.sql').read_text() and len(rows)==second['clips']
        assert 'S_BAD' in (OUTPUT_DIR/'errors.jsonl').read_text();print('metadata mock passed',first)
    finally:
        globals().update(original)
mock_test()


In [ ]:
# Optional FFmpeg portion of the mandatory mock test.
def mock_materialize_test()->None:
    global INPUT_DIR,OUTPUT_DIR,SHOT_FILE,MATERIALIZE_FILES,VIDEOS_FILE,CHECKPOINT,ERRORS,MANIFEST
    names=('INPUT_DIR','OUTPUT_DIR','SHOT_FILE','MATERIALIZE_FILES','VIDEOS_FILE','CHECKPOINT','ERRORS','MANIFEST')
    original={name:globals()[name] for name in names}
    try:
        command(['ffmpeg','-version']); command(['ffprobe','-version'])
        root=Path('/tmp/clip_materialize_mock');shutil.rmtree(root,ignore_errors=True);root.mkdir();INPUT_DIR=root;OUTPUT_DIR=root/'out';OUTPUT_DIR.mkdir();SHOT_FILE=root/'shot.csv';CHECKPOINT=OUTPUT_DIR/'checkpoint.json';ERRORS=OUTPUT_DIR/'errors.jsonl';MANIFEST=OUTPUT_DIR/'clip_manifest.jsonl';VIDEOS_FILE=''
        video=INPUT_DIR/'synthetic.mp4';command(['ffmpeg','-y','-f','lavfi','-i','testsrc2=size=64x64:rate=10:duration=12','-c:v','libx264','-pix_fmt','yuv420p',str(video)])
        SHOT_FILE.write_text(f'shot_id,video_id,shot_index,start_ms,end_ms,start_frame_idx,end_frame_idx,video_path\nL1_V001_S0001,v3,0,0,11000,0,110,{video}\n')
        MATERIALIZE_FILES=True;summary=run_pipeline();files=sorted((OUTPUT_DIR/'clips'/'v3').glob('*.mp4'))
        assert len(files)>=2 and all(p.stat().st_size>0 and abs(duration_ms(p)-10000)<=150 for p in files[:1])
        print('materialization mock passed',summary)
    finally:
        globals().update(original)
# Optional: call mock_materialize_test() manually; it creates a synthetic MP4.


## Vận hành

Attach `shot.csv`; chỉ khi `MATERIALIZE_FILES=True` mới attach video qua `video_path` hoặc `videos.csv`. Chỉnh range rồi **Run All**, tải toàn bộ `OUTPUT_DIR`, import bảng `shot` trước, sau đó chạy `insert_clipwindows.sql`.
